### KIKO_Real-Time Detection and Filtering of Abusive Content

Feature extraction


In [23]:
import pandas as pd
import numpy as np

In [24]:
df1 = pd.read_csv('D:\\ML CODES\\IBM\\Train_Cleaned\\Airline_train_cleaned.csv')
df2 = pd.read_csv('D:\\ML CODES\\IBM\\Train_Cleaned\\text.tweeet_cleaned.csv')
df3 = pd.read_csv('D:\\ML CODES\\IBM\\Train_Cleaned\\train_cleaned.csv')
df4 = pd.read_csv('D:\\ML CODES\\IBM\\Train_Cleaned\\train_tweet_cleaned.csv')

#Concatenating the dataframes

df = pd.concat([df1, df2, df3, df4], ignore_index=True)

In [38]:
df.head()

,sentiment,text
0,neutral,What said
1,positive,plus youve added commercials to the experienc...
2,neutral,I didnt today Must mean I need to take anothe...
3,negative,its really aggressive to blast obnoxious ente...
4,negative,and its a really big bad thing about it


In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 105140 entries, 0 to 105462
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   sentiment  105140 non-null  object
 1   text       105140 non-null  object
dtypes: object(2)
memory usage: 2.4+ MB


In [36]:
print(df['sentiment'].value_counts())

sentiment
positive    49825
negative    38138
neutral     17177
Name: count, dtype: int64


In [30]:
df.duplicated().sum()

0

In [29]:
df.drop_duplicates(inplace=True)

In [35]:
df.dropna(inplace=True)

In [40]:
# --- Step 1: Import necessary libraries ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
import warnings
warnings.filterwarnings('ignore')

In [41]:
# --- Step 2: Prepare features (X) and labels (y) ---
X = df['text']
y = df['sentiment']

print(f"Total samples: {X.shape[0]}")
print(f"Label distribution:\n{y.value_counts()}")

Total samples: 105140
Label distribution:
sentiment
positive    49825
negative    38138
neutral     17177
Name: count, dtype: int64


In [42]:
# --- Step 3: Split into training (80%) and testing (20%) sets ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # preserve label distribution in both splits
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")

Training samples: 84112
Testing samples:  21028


In [43]:
# --- Step 4: Build the Pipeline (TF-IDF → LinearSVC) ---
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(sublinear_tf=True, strip_accents='unicode')),
    ('clf', LinearSVC(max_iter=10000, dual='auto'))
])

In [44]:
# --- Step 5: Define hyperparameter grid for GridSearchCV ---
param_grid = {
    # TF-IDF parameters
    'tfidf__ngram_range': [(1, 1), (1, 2)],       # unigrams vs unigrams+bigrams
    'tfidf__max_df': [0.85, 0.90, 0.95],           # ignore terms appearing in >X% of docs
    'tfidf__min_df': [2, 5],                        # ignore terms appearing in <X docs

    # LinearSVC regularization parameter
    'clf__C': [0.1, 0.5, 1.0, 2.0],
}

print(f"Total combinations to search: "
      f"{np.prod([len(v) for v in param_grid.values()])}")

Total combinations to search: 48


In [ ]:
# --- Step 6: Run GridSearchCV with 5-fold cross-validation ---
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,                   # 5-fold stratified cross-validation
    scoring='f1_weighted',  # weighted F1 handles class imbalance
    n_jobs=-1,              # use all CPU cores
    verbose=2,
    refit=True              # refit best model on entire training set
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 48 candidates, totalling 240 fits


In [ ]:
# --- Step 7: Print the best parameters and best CV score ---
print("=" * 60)
print("KIKO - Best Hyperparameters Found")
print("=" * 60)
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nBest Cross-Validation F1 (weighted): {grid_search.best_score_:.4f}")

In [ ]:
# --- Step 8: Evaluate on the test set ---
y_pred = grid_search.predict(X_test)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall    = recall_score(y_test, y_pred, average='weighted')
f1        = f1_score(y_test, y_pred, average='weighted')
cm        = confusion_matrix(y_test, y_pred, labels=['Positive', 'Neutral', 'Negative'])

print("=" * 60)
print("KIKO - Test Set Evaluation Results")
print("=" * 60)
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print()

# Detailed per-class report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("Confusion Matrix (rows=actual, cols=predicted):")
print("Labels: [Positive, Neutral, Negative]")
print(cm)

In [ ]:
# --- Step 9: Visualize Confusion Matrix ---
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Positive', 'Neutral', 'Negative'],
    yticklabels=['Positive', 'Neutral', 'Negative'],
    ax=ax
)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('Actual Label', fontsize=12)
ax.set_title('KIKO - Confusion Matrix (LinearSVC + TF-IDF)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- Step 10: Save the best model for later use ---
import joblib

joblib.dump(grid_search.best_estimator_, 'D:\\ML CODES\\IBM\\kiko_model.pkl')
print("Best model saved to: D:\\ML CODES\\IBM\\kiko_model.pkl")